##### MAP REPRESENTATION

In [ ]:
# https://figshare.com/articles/dataset/_b_Mexico_GeoJSON_States_and_Municipalities_2024_b_/31236322
import json
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import unicodedata
import warnings

warnings.filterwarnings('ignore')

In [ ]:
# Open map and data file
data_path = './final_datasets'
with open(f'{data_path}/states_full.geojson', encoding='utf-8') as f:
    states = json.load(f)
state_dict = {feature['properties']['NOMGEO'].upper(): feature['properties']['CVE_ENT'] for feature in states['features']}
state_dict = {unicodedata.normalize('NFKD', key).encode('ASCII', 'ignore').decode('ASCII'): value.strip() for key, value in state_dict.items()}
state_dict.update({'COAHUILA':'05', 'MICHOACAN':'16', 'VERACRUZ':'30'})

dataset = pd.read_parquet(f'{data_path}/conteos_censales.parquet')
filtered_data = dataset.query('AELEC == 2024').groupby('EDONOM').agg({'LN':'sum', 'SV':'sum', 'NV':'sum', 'NS':'sum'}).reset_index()
filtered_data.insert(0, 'CVE_ENT', filtered_data['EDONOM'].map(state_dict))
filtered_data.columns = ['CVE_ENT','Estado','ListaNominal','SíVotaron','NoVotaron','NoEspecificado']

In [ ]:
# Plot general KPIs
fig = go.Figure()

# Roll call
fig.add_trace(go.Indicator(
    mode='number',
    value=filtered_data.agg({'ListaNominal':'sum'}).item(),
    domain={'x':[0, 0.2], 'y':[0.6, 1]},
    title={'text':'Lista Nominal', 'font':{'size':20, 'color':'#6b1590'}},
    number={'font': {'size': 55, 'color': '#6b1590'}}
))

# Did vote
fig.add_trace(go.Indicator(
    mode='number',
    value=filtered_data.agg({'SíVotaron':'sum'}).item(),
    domain={'x':[0.25, 0.45], 'y':[0.6, 1]},
    title={'text':'Sí Votaron', 'font':{'size':20, 'color':'#6b1590'}},
    number={'font': {'size': 55, 'color': '#6b1590'}}
))

# Did not vote
fig.add_trace(go.Indicator(
    mode='number',
    value=filtered_data.agg({'NoVotaron':'sum'}).item(),
    domain={'x':[0.5, 0.7], 'y':[0.6, 1]},
    title={'text':'No Votaron', 'font':{'size':20, 'color':'#6b1590'}},
    number={'font': {'size': 55, 'color': '#6b1590'}}))

# Not specified
fig.add_trace(go.Indicator(
    mode='number',
    value=filtered_data.agg({'NoEspecificado':'sum'}).item(),
    domain={'x':[0.75, 0.95], 'y':[0.6, 1]},
    title={'text':'Votos No Especificados', 'font':{'size':20, 'color':'#6b1590'}},
    number={'font': {'size': 55, 'color': '#6b1590'}}))

fig.show()

In [ ]:
# Plot interactive map
fig = px.choropleth(
    filtered_data,
    geojson=states,
    locations='CVE_ENT',
    featureidkey='properties.CVE_ENT',
    color='Valor',
    labels={'Valor':'Votos'},
    color_continuous_scale=px.colors.diverging.PRGn,
    hover_name='Estado',
    hover_data={
        'CVE_ENT': False,
        'Valor': ':,',
        'Estado': False
    }
)

fig.update_layout(
    margin={'r':0, 't':0, 'l':0, 'b':0},
    hoverlabel=dict(
        bgcolor='white',
        bordercolor='gray',
        font=dict(
            family='Arial',
            size=12,
            color='black'
        )
    )
)

fig.update_geos(
    visible=False,
    showframe=False,
    fitbounds='locations'
)

fig.show()

Other ideas:
- Basic charts
- Table of full results
- Early votting and pretial detention data
- Comparative analysis by subgroups
- Geographical data segmentation